# Reading another project's results

`labdata` lets one project read result files produced by another, without
submodules, without cloning, and without downloading anything by hand.

A repository publishes by committing a `labdata.yml` in the `results/`
directory at its root, naming the files it wants others to see and saying what
each one holds. Everything else in `results/` stays private to that project.

Repositories are read wherever they are: clones on this machine, clones on a
server reached over ssh, and repositories on GitHub read over the API without
being cloned at all.

In [1]:
import pandas as pd
import vscodenb
import labdata

### `labdata.yml`

To register files or directories as data available to `labdata`, add a `labdata.yml` to the repository `results`. Keys are file names, paths or globs and values say what the file holds. You can list parquet folders as if they were files, since they load as such.

```yml
files:
  hits.csv: Genome-wide association hits, p < 5e-8
  qc/summary.csv: Per-sample genotyping quality
  by_chrom: Per-chromosome effect sizes, one file per chromosome
```

A key beginning with `/` is a path from the repository root rather than from the
`labdata.yml`, which publishes a file that is kept elsewhere in the repository.
Any tracked file can be named this way; `..` cannot be used to climb out.

```yml
files:
  hits.csv: In the results directory, as usual
  /data/reference/samples.csv: Somewhere else in the repository
  /data/raw/*.tsv: A pattern, matched against the path from the root
```


## Config

Roots on another machine are read over ssh. Where the host asks for a key
passphrase or a second factor, ssh prompts for it at the terminal, once, and the
shared connection covers the commands that follow. In a notebook the prompt opens at the top of the
window, as any `getpass` in a cell does, and what you type goes to ssh. To be
asked once a day rather than once a session, give the host a
`ControlMaster`/`ControlPersist` block in `~/.ssh/config`; labdata reads through
the connection you already opened.

`labdata` configuration is read from a config file that can be generated with `labdata config --init` and shown using:

In [2]:
! labdata config


/Users/kmt/sandbox/labdata/.pixi/envs/default/lib/python3.13/site-packages/labdata/cli.py:540: UserWarning: /Users/kmt/.config/labdata/config.toml: `results_dirs` is now `labdata_dirs`
  cfg = Config.load(path)
config file: /Users/kmt/.config/labdata/config.toml
  roots = ['kmt@login.genome.au.dk:xy-drive/people/kmt']
  owners = ['munch-group']
  repos = []
  labdata_dirs = ['results', 'data']
  include = ['*.csv', '*.tsv', '*.txt', '*.parquet', '*.pq', '*.h5', '*.hdf', '*.hdf5', '*.store', '*.json', '*.jsonl', '*.xlsx', '*.bed', '*.gff', '*.vcf', '*.vcf.gz', '*.pkl', '*.pickle', '*.npy', '*.npz', '*.feather', '*.zarr']
  exclude = ['*.png', '*.pdf', '*.svg', '*.html', '*.md', '.gitkeep', '*.log']
  min_bytes = 0
  max_bytes = 0
cache: 135 objects, 6.1G


You can also generate one on the fly in the notebook and pass along to each labdata function:

In [10]:
cfg = labdata.config.Config(
    labdata_dirs=['results', 'steps/data'],
    owners=["munch-group", ], # search all repos under these accounts
    # owners=["munch-group", "erikfogh" ], # search all repos under these accounts
    # repos=['munch-group/relate1Kgenomes'],
    # Root folders to search for local github repos:
    # (remote server access requires passwordless acess with ssh keys)
    roots = ["kmt@login.genome.au.dk:xy-drive/people/kmt", ] 
    # roots = ["~/Documents/projects", "kmt@login.genome.au.dk:xy-drive/people/kmt", ] 
    # include=[],
    # exclude=[],
)
df = labdata.refresh(cfg=cfg)

scanning clones:   0%|          | 0/12 [00:00<?, ?repo/s]

reading GitHub:   0%|          | 0/155 [00:00<?, ?repo/s]

In [12]:
from pandas.api.types import is_object_dtype

class nice:

    def __rlshift__(self, df):
        "Left align columns of params frame: df << nice()"
        s = df.style
        s.set_table_styles(
            {c: [{'selector': '', 'props': [('text-align', 'left')]}] 
                 for c in df.columns if is_object_dtype(df[c])},
            overwrite=False
        )
        display(s)

In [13]:
labdata.list(brief=True, cfg=cfg).set_index(['owner', 'repo', 'name', 'size']) << nice()

In [14]:
tmp_path = labdata.get("atlas-variant-ages", "atlas_variant_ages.parquet", cfg=cfg)

Add version="b9822e9e86988b04b3eec8edf7f8aa779e3c609c" to pin this version.


In [15]:
pd.read_parquet(tmp_path).head()

: 

## Refresh

In [2]:
labdata.refresh()

scanning clones:   0%|          | 0/63 [00:00<?, ?repo/s]

reading GitHub:   0%|          | 0/155 [00:00<?, ?repo/s]

,repo,path,description,date,bytes,tags,lfs
0,munch-group/relate1Kgenomes,results/snp_data.parquet,P-values for all chromosomes and populations,2026-08-31,41270846,,False
1,munch-group/tree-stats,results/dummy.csv,Some dummy csv file,2026-08-30,19,,False


## List

`labdata.list()` is the python side of `labdata list`.

In [5]:
labdata.list().groupby('repo')

/Users/kmt/sandbox/labdata/.pixi/envs/default/lib/python3.13/site-packages/labdata/core.py:961: UserWarning: /Users/kmt/.config/labdata/config.toml: `results_dirs` is now `labdata_dirs`
  cfg = cfg or Config.load()


KeyboardInterrupt: 

In [ ]:
labdata.list(version=True, cfg=cfg)

,repo,path,description,date,bytes,tags,lfs,version
0,munch-group/relate1Kgenomes,results/snp_data.parquet,P-values for all chromosomes and populations,2026-08-31,41270846,,False,ba3c72b3b259a1b2d47ab3aaf5415a112bd644cd
1,munch-group/tree-stats,results/dummy.csv,Some dummy csv file,2026-08-30,19,,False,770170789d73428efb0193d4cf04a353587db9cd


`scratch_notes.csv` is committed but absent: it is not named in the manifest. `by_chrom` is one row rather than three, because the manifest names the directory — `parts` says how many files are behind it. The version is left out by default, since it is a full commit sha and every file a repository publishes carries the same one. Ask for it when you want it:

In [ ]:
labdata.list(version=True, cfg=cfg)[['repo', 'path', 'version']]

,repo,path,version
0,munch-group/relate1Kgenomes,results/snp_data.parquet,ba3c72b3b259a1b2d47ab3aaf5415a112bd644cd
1,munch-group/tree-stats,results/dummy.csv,770170789d73428efb0193d4cf04a353587db9cd


## Repos

In [ ]:
labdata.repos()

,repo,files,bytes,latest
0,munch-group/relate1Kgenomes,1,41270846,2026-08-31
1,munch-group/tree-stats,1,19,2026-08-30


## Get

`labdata.get()` returns a local path, so it goes straight into `read_csv`,
`read_parquet`, `h5py.File`, or anything else that takes a path.

In [9]:
tmp_path = labdata.get("munch-group/relate1Kgenomes", 
                       "results/snp_data.parquet")

Add version="ba3c72b3b259a1b2d47ab3aaf5415a112bd644cd" to pin this version.


In [10]:
df = pd.read_parquet(tmp_path)
df.head()

,chrom,pos,neglog10p,population,region
0,chrX,2781514,0.060365,ASW,Africa
1,chrX,2781584,0.372703,ASW,Africa
2,chrX,2781604,0.156910,ASW,Africa
3,chrX,2781635,1.034000,ASW,Africa
4,chrX,2781927,0.630687,ASW,Africa


In [10]:
# silent, because this is still the current version
labdata.get("munch-group/relate1Kgenomes", 
            "results/snp_data.parquet", 
            version="ba3c72b3b259a1b2d47ab3aaf5415a112bd644cd")

PosixPath('/Users/kmt/.cache/labdata/files/munch-group__relate1Kgenomes/ba3c72b3b259a1b2d47ab3aaf5415a112bd644cd/results/snp_data.parquet')

Notice what it printed. Without a version it takes the current one and tells
you the sha that pins it, together with the call that does so — paste that back
into the cell and the notebook reads the same bytes next year.

A pinned call is silent, and the sha is a full one, so it also means something
to `git show` or a GitHub URL without labdata in hand.

In [ ]:
tmp_path = labdata.get("munch-group/relate1Kgenomes", 
                       "results/snp_data.parquet", 
                       "ba3c72b3b259a1b2d47ab3aaf5415a112bd644cd")

The file name is enough when it is unambiguous; otherwise give as much of the
path as it takes.

In [6]:
labdata.get("relate1Kgenomes", "snp_data.parquet")

Add version="ba3c72b3b259a1b2d47ab3aaf5415a112bd644cd" to pin this version.


PosixPath('/Users/kmt/.cache/labdata/files/munch-group__relate1Kgenomes/ba3c72b3b259a1b2d47ab3aaf5415a112bd644cd/results/snp_data.parquet')

In [ ]:
pd.read_parquet(labdata.get("relate1Kgenomes", "snp_data.parquet"))

Add version="ba3c72b3b259a1b2d47ab3aaf5415a112bd644cd" to pin this version.


,chrom,pos,neglog10p,population,region
0,chrX,2781514,0.060365,ASW,Africa
1,chrX,2781584,0.372703,ASW,Africa
2,chrX,2781604,0.156910,ASW,Africa
3,chrX,2781635,1.034000,ASW,Africa
4,chrX,2781927,0.630687,ASW,Africa
...,...,...,...,...,...
5378318,chrX,155697006,0.271943,ACB,Caribia
5378319,chrX,155697134,0.066021,ACB,Caribia
5378320,chrX,155697920,0.542909,ACB,Caribia
5378321,chrX,155699015,0.050564,ACB,Caribia


For a real partitioned parquet table the last step is simply
a

pd.read_parquet(labdata.get("relate1Kgenomes", "snp_data.parquet"))
```

since `read_parquet` takes a directory. Nothing about `.parquet` is special to
labdata — naming a directory in the manifest is what makes it a dataset, so
`.zarr` and anything else shaped that way work the same.

## History

The catalog stamps every file with the repository's current commit.
`labdata.versions()` shows the commits in which the file itself changed, which
is the useful set to pin.

In [13]:
labdata.versions("x-gwas", "hits.csv")

,version,date,bytes,parts,tags,subject
0,dc55b1f01fdd98af9572fb976527bb7ccd30635b,2026-08-31,61,0,,publish association results


## When nothing shows up

An empty catalog has several ordinary causes and they look alike from outside.
`labdata.diagnose()` says which it is, reading only local git.

In [14]:
print("\n".join(labdata.diagnose(labdata.Config(roots=["~/no-such-place"]))))

  ~/no-such-place: no such directory
  github: no owners or repos configured, so nothing is read from GitHub
